# Credit Approval — Model Training
Training and comparing 5 base models + stacking ensemble with 10-fold CV and SMOTEENN.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.utils import load_raw_data, TARGET_COL
from src.preprocessing import build_preprocessing_pipeline, get_feature_names
from src.training import run_all_base_models, run_optuna_hpo, train_stacking_ensemble
from src.evaluation import build_results_table

Path("../outputs/models").mkdir(parents=True, exist_ok=True)
Path("../outputs/reports").mkdir(parents=True, exist_ok=True)

df = load_raw_data("../data/raw/crx.data")
X_raw = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

X_tr_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.15, stratify=y, random_state=42
)

preproc = build_preprocessing_pipeline()
X_train = preproc.fit_transform(X_tr_raw, y_train)
X_test  = preproc.transform(X_test_raw)
feature_names = get_feature_names(preproc)

joblib.dump(preproc, "../outputs/models/preprocessor.pkl")
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

In [ ]:
# Base model CV
print("Training base models with 10-fold CV + SMOTEENN...")
cv_results = run_all_base_models(X_train, y_train)

In [ ]:
# HPO — set n_trials low for notebook (increase for production runs)
print("Running Optuna HPO (30 trials each)...")
best_params = run_optuna_hpo(X_train, y_train, n_trials_lgb=30, n_trials_xgb=30)
print("Best params:")
for model, params in best_params.items():
    print(f"  {model}: {params}")

In [ ]:
# Stacking ensemble
lgb_best = best_params.get("LightGBM")
xgb_best = best_params.get("XGBoost")
ensemble, stack_metrics = train_stacking_ensemble(X_train, y_train,
                                                   lgb_params=lgb_best,
                                                   xgb_params=xgb_best)
joblib.dump(ensemble, "../outputs/models/stacking_ensemble.pkl")

In [ ]:
# Results table
all_cv = {**cv_results, "StackingEnsemble": stack_metrics}
results_df = build_results_table(all_cv)
print(results_df.to_string(index=False))